**TAHAP 1 :** IMPORT LIBRARY

In [5]:
import warnings
warnings.filterwarnings("ignore")

#Random Seed

import random

SEED = 42

random.seed(SEED)

#Numerical

import numpy as np
import pandas as pd

np.random.seed(SEED)

#Visualization

import matplotlib.pyplot as plt

#Machine Learning

from sklearn.model_selection import (
    RepeatedStratifiedKFold,
    cross_validate,
    cross_val_predict
)

from sklearn.preprocessing import LabelEncoder

from sklearn.feature_extraction.text import (
    TfidfVectorizer
)

from sklearn.pipeline import Pipeline

from sklearn.naive_bayes import MultinomialNB

from sklearn.svm import LinearSVC

#Evaluation

from sklearn.metrics import (

    accuracy_score,

    precision_score,

    recall_score,

    f1_score,

    classification_report,

    confusion_matrix

)

#Stastical Test

from scipy.stats import (
    ttest_rel,
    wilcoxon
)

#TensorFlow Random Seed

try:

    import tensorflow as tf

    tf.random.set_seed(SEED)

    print("TensorFlow random seed berhasil diatur.")

except:

    print("TensorFlow belum dimuat.")

print("=" * 70)
print("TAHAP 1 : IMPORT LIBRARY")
print("=" * 70)

print(f"Random Seed : {SEED}")

print("Library berhasil dimuat.")

TensorFlow random seed berhasil diatur.
TAHAP 1 : IMPORT LIBRARY
Random Seed : 42
Library berhasil dimuat.


**TAHAP 2 :** LOAD DATASET

In [6]:
print("\nMemuat dataset...")

df = pd.read_csv("/content/dataset_trenfashion.csv")

print("Jumlah data :", len(df))

print("\nUkuran dataset :", df.shape)

print("\nKolom dataset :")

print(df.columns.tolist())

print("\n5 Data pertama :")

display(df.head())

print("\nCek Missing Value")

print(df.isnull().sum())

assert len(df) == 20000, "Jumlah dataset tidak sesuai."

assert "ulasan" in df.columns, "Kolom text tidak ditemukan."

assert "sentimen" in df.columns, "Kolom sentiment tidak ditemukan."


Memuat dataset...
Jumlah data : 20000

Ukuran dataset : (20000, 2)

Kolom dataset :
['ulasan', 'sentimen']

5 Data pertama :


,ulasan,sentimen
0,buat aku pribadi kurang menarik menurutku,Negatif
1,warnanya lumayan sih,Netral
2,sejujurnya terlalu biasa tapi jadi suka,Positif
3,awalnya oke tapi masih standar,Netral
4,looknya lumayan jujur,Positif



Cek Missing Value
ulasan      0
sentimen    0
dtype: int64


**TAHAP 3 :** CEK MISSING VALUE

In [8]:
print("=" * 70)
print("Pemeriksaan Dataset")
print("=" * 70)

print("\nInformasi Dataset")

df.info()

print("\nMissing Value")

print(df.isnull().sum())

print("\nJumlah Data Duplikat")

print(df.duplicated().sum())

print("\nDistribusi Label")

print(df["sentimen"].value_counts())

Pemeriksaan Dataset

Informasi Dataset
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   ulasan    20000 non-null  object
 1   sentimen  20000 non-null  object
dtypes: object(2)
memory usage: 312.6+ KB

Missing Value
ulasan      0
sentimen    0
dtype: int64

Jumlah Data Duplikat
11078

Distribusi Label
sentimen
Negatif    7000
Positif    7000
Netral     6000
Name: count, dtype: int64


In [10]:
print(df.columns.tolist())

['ulasan', 'sentimen', 'clean_text']


**TAHAP 4 :** TEXT PREPROCESSING

In [11]:
import re

def clean_text(text):

    if pd.isna(text):

        return ""

    text = str(text).lower()

    text = re.sub(
        r"[^a-zA-Z0-9\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


# preprocessing

df["clean_text"] = df["ulasan"].apply(clean_text)

print("=" * 70)
print("Preprocessing selesai")
print("=" * 70)

print("\nJumlah clean text kosong")

print(
    (df["clean_text"] == "").sum()
)

print("\nContoh hasil preprocessing")

display(
    df[
        [
            "ulasan",
            "clean_text",
            "sentimen"
        ]
    ].head(10)
)

Preprocessing selesai

Jumlah clean text kosong
0

Contoh hasil preprocessing


,ulasan,clean_text,sentimen
0,buat aku pribadi kurang menarik menurutku,buat aku pribadi kurang menarik menurutku,Negatif
1,warnanya lumayan sih,warnanya lumayan sih,Netral
2,sejujurnya terlalu biasa tapi jadi suka,sejujurnya terlalu biasa tapi jadi suka,Positif
3,awalnya oke tapi masih standar,awalnya oke tapi masih standar,Netral
4,looknya lumayan jujur,looknya lumayan jujur,Positif
5,awalnya kurang yakin tapi ternyata kereen menu...,awalnya kurang yakin tapi ternyata kereen menu...,Positif
6,modelnya oke tapi tetap terlihat menarik,modelnya oke tapi tetap terlihat menarik,Positif
7,kombinasi warnanya membosankan,kombinasi warnanya membosankan,Negatif
8,buat saya pribadi kurang cocok tapi jadi suka sih,buat saya pribadi kurang cocok tapi jadi suka sih,Positif
9,awalnya lumayan tapi ga buruk juga jujur,awalnya lumayan tapi ga buruk juga jujur,Netral


**TAHAP 5 :** LABEL ENCODING & CROSS VALIDATION

In [12]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

df["label"] = label_encoder.fit_transform(df["sentimen"])

X = df["clean_text"]

y = df["label"]

print("=" * 70)
print("Label Encoding")
print("=" * 70)

mapping = dict(

    zip(

        label_encoder.classes_,

        label_encoder.transform(

            label_encoder.classes_

        )

    )

)

print("\nMapping Label")

for k, v in mapping.items():

    print(f"{k} -> {v}")

print()

print("Distribusi Label")

print(

    df["label"]

    .value_counts()

    .sort_index()

)

print()

print("Jumlah Data :", len(df))

assert len(X) == len(y), "Jumlah X dan y tidak sama."

Label Encoding

Mapping Label
Negatif -> 0
Netral -> 1
Positif -> 2

Distribusi Label
label
0    7000
1    6000
2    7000
Name: count, dtype: int64

Jumlah Data : 20000


T**AHAP 6 :** MEMBANGUN PIPELINE NAIVE BAYES

In [13]:
from sklearn.model_selection import train_test_split

def split_data(random_state=SEED):

    X_train, X_test, y_train, y_test = train_test_split(

        X,

        y,

        test_size=0.20,

        stratify=y,

        random_state=random_state

    )

    return X_train, X_test, y_train, y_test

print("=" * 70)
print("Fungsi split_data() berhasil dibuat.")
print("=" * 70)

X_train, X_test, y_train, y_test = split_data()

print()

print("Training :", len(X_train))

print("Testing  :", len(X_test))

assert len(X_train) == len(y_train)

assert len(X_test) == len(y_test)

print("\nDistribusi Training")

print(y_train.value_counts().sort_index())

print("\nDistribusi Testing")

print(y_test.value_counts().sort_index())

Fungsi split_data() berhasil dibuat.

Training : 16000
Testing  : 4000

Distribusi Training
label
0    5600
1    4800
2    5600
Name: count, dtype: int64

Distribusi Testing
label
0    1400
1    1200
2    1400
Name: count, dtype: int64


**TAHAP 7 :** EVALUASI NAIVE BAYES

In [14]:
random_states = list(range(SEED, SEED + 30))

nb_accuracy = []
nb_precision = []
nb_recall = []
nb_f1 = []

print("=" * 70)
print("EKSPERIMEN NAIVE BAYES")
print("=" * 70)

for rs in random_states:

    X_train, X_test, y_train, y_test = split_data(rs)

    tfidf = TfidfVectorizer(

        max_features=5000,

        ngram_range=(1, 2)

    )

    X_train_tfidf = tfidf.fit_transform(X_train)

    X_test_tfidf = tfidf.transform(X_test)

    model = MultinomialNB()

    model.fit(

        X_train_tfidf,

        y_train

    )

    nb_pred = model.predict(

        X_test_tfidf

    )

    acc = accuracy_score(
        y_test,
        nb_pred
    )

    pre = precision_score(
        y_test,
        nb_pred,
        average="weighted"
    )

    rec = recall_score(
        y_test,
        nb_pred,
        average="weighted"
    )

    f1 = f1_score(
        y_test,
        nb_pred,
        average="weighted"
    )

    nb_accuracy.append(acc)
    nb_precision.append(pre)
    nb_recall.append(rec)
    nb_f1.append(f1)

    print(
        f"Seed {rs} | "
        f"Accuracy = {acc:.4f}"
    )

#Hasil Eksperimen

nb_result = pd.DataFrame({

    "Seed": random_states,

    "Accuracy": nb_accuracy,

    "Precision": nb_precision,

    "Recall": nb_recall,

    "F1-score": nb_f1

})

print("\n")
print("=" * 70)
print("RATA-RATA NAIVE BAYES")
print("=" * 70)

print(f"Accuracy      : {np.mean(nb_accuracy):.4f}")
print(f"Precision     : {np.mean(nb_precision):.4f}")
print(f"Recall        : {np.mean(nb_recall):.4f}")
print(f"F1-score      : {np.mean(nb_f1):.4f}")
print(f"Std Accuracy  : {np.std(nb_accuracy):.4f}")

print("\n5 Hasil Pertama")

display(nb_result.head())

EKSPERIMEN NAIVE BAYES
Seed 42 | Accuracy = 0.9435
Seed 43 | Accuracy = 0.9407
Seed 44 | Accuracy = 0.9450
Seed 45 | Accuracy = 0.9490
Seed 46 | Accuracy = 0.9410
Seed 47 | Accuracy = 0.9395
Seed 48 | Accuracy = 0.9445
Seed 49 | Accuracy = 0.9475
Seed 50 | Accuracy = 0.9447
Seed 51 | Accuracy = 0.9495
Seed 52 | Accuracy = 0.9510
Seed 53 | Accuracy = 0.9483
Seed 54 | Accuracy = 0.9477
Seed 55 | Accuracy = 0.9375
Seed 56 | Accuracy = 0.9485
Seed 57 | Accuracy = 0.9495
Seed 58 | Accuracy = 0.9450
Seed 59 | Accuracy = 0.9390
Seed 60 | Accuracy = 0.9480
Seed 61 | Accuracy = 0.9500
Seed 62 | Accuracy = 0.9443
Seed 63 | Accuracy = 0.9427
Seed 64 | Accuracy = 0.9427
Seed 65 | Accuracy = 0.9473
Seed 66 | Accuracy = 0.9480
Seed 67 | Accuracy = 0.9537
Seed 68 | Accuracy = 0.9433
Seed 69 | Accuracy = 0.9453
Seed 70 | Accuracy = 0.9443
Seed 71 | Accuracy = 0.9427


RATA-RATA NAIVE BAYES
Accuracy      : 0.9455
Precision     : 0.9455
Recall        : 0.9455
F1-score      : 0.9454
Std Accuracy  : 0.003

,Seed,Accuracy,Precision,Recall,F1-score
0,42,0.94350,0.943479,0.94350,0.943452
1,43,0.94075,0.940810,0.94075,0.940762
2,44,0.94500,0.944984,0.94500,0.944986
3,45,0.94900,0.949012,0.94900,0.948963
4,46,0.94100,0.940957,0.94100,0.940967


**TAHAP 8 :** EKSPERIMEN SVM

In [15]:
#Support Vector Machine

random_states = list(range(SEED, SEED + 30))

svm_accuracy = []
svm_precision = []
svm_recall = []
svm_f1 = []

print("=" * 70)
print("EKSPERIMEN SUPPORT VECTOR MACHINE")
print("=" * 70)

for rs in random_states:

    X_train, X_test, y_train, y_test = split_data(rs)

    tfidf = TfidfVectorizer(

        max_features=5000,

        ngram_range=(1, 2)

    )

    X_train_tfidf = tfidf.fit_transform(X_train)

    X_test_tfidf = tfidf.transform(X_test)

    model = LinearSVC(

        random_state=rs

    )

    model.fit(

        X_train_tfidf,

        y_train

    )

    svm_pred = model.predict(

        X_test_tfidf

    )

    acc = accuracy_score(
        y_test,
        svm_pred
    )

    pre = precision_score(
        y_test,
        svm_pred,
        average="weighted"
    )

    rec = recall_score(
        y_test,
        svm_pred,
        average="weighted"
    )

    f1 = f1_score(
        y_test,
        svm_pred,
        average="weighted"
    )

    svm_accuracy.append(acc)
    svm_precision.append(pre)
    svm_recall.append(rec)
    svm_f1.append(f1)

    print(
        f"Seed {rs} | "
        f"Accuracy = {acc:.4f}"
    )

#Hasil Eksperimen

svm_result = pd.DataFrame({

    "Seed": random_states,

    "Accuracy": svm_accuracy,

    "Precision": svm_precision,

    "Recall": svm_recall,

    "F1-score": svm_f1

})

print("\n")
print("=" * 70)
print("RATA-RATA SUPPORT VECTOR MACHINE")
print("=" * 70)

print(f"Accuracy      : {np.mean(svm_accuracy):.4f}")
print(f"Precision     : {np.mean(svm_precision):.4f}")
print(f"Recall        : {np.mean(svm_recall):.4f}")
print(f"F1-score      : {np.mean(svm_f1):.4f}")
print(f"Std Accuracy  : {np.std(svm_accuracy):.4f}")

print("\n5 Hasil Pertama")

display(svm_result.head())

EKSPERIMEN SUPPORT VECTOR MACHINE
Seed 42 | Accuracy = 0.9495
Seed 43 | Accuracy = 0.9493
Seed 44 | Accuracy = 0.9495
Seed 45 | Accuracy = 0.9500
Seed 46 | Accuracy = 0.9415
Seed 47 | Accuracy = 0.9455
Seed 48 | Accuracy = 0.9470
Seed 49 | Accuracy = 0.9517
Seed 50 | Accuracy = 0.9473
Seed 51 | Accuracy = 0.9513
Seed 52 | Accuracy = 0.9510
Seed 53 | Accuracy = 0.9563
Seed 54 | Accuracy = 0.9497
Seed 55 | Accuracy = 0.9405
Seed 56 | Accuracy = 0.9523
Seed 57 | Accuracy = 0.9493
Seed 58 | Accuracy = 0.9445
Seed 59 | Accuracy = 0.9453
Seed 60 | Accuracy = 0.9483
Seed 61 | Accuracy = 0.9497
Seed 62 | Accuracy = 0.9460
Seed 63 | Accuracy = 0.9467
Seed 64 | Accuracy = 0.9465
Seed 65 | Accuracy = 0.9493
Seed 66 | Accuracy = 0.9505
Seed 67 | Accuracy = 0.9590
Seed 68 | Accuracy = 0.9457
Seed 69 | Accuracy = 0.9433
Seed 70 | Accuracy = 0.9493
Seed 71 | Accuracy = 0.9513


RATA-RATA SUPPORT VECTOR MACHINE
Accuracy      : 0.9486
Precision     : 0.9488
Recall        : 0.9486
F1-score      : 0.9485

,Seed,Accuracy,Precision,Recall,F1-score
0,42,0.94950,0.949982,0.94950,0.949372
1,43,0.94925,0.949523,0.94925,0.949153
2,44,0.94950,0.949490,0.94950,0.949477
3,45,0.95000,0.950350,0.95000,0.949893
4,46,0.94150,0.941835,0.94150,0.941369


**TAHAP 9 :** REKAPITULASI HASIL EKSPERIMEN

In [16]:
#Hasil Eksperimen

result_df = pd.DataFrame({

    "Random State": random_states,

    "Naive Bayes": nb_accuracy,

    "SVM": svm_accuracy

})

result_df["Difference"] = (

    result_df["SVM"]

    - result_df["Naive Bayes"]

)

print("=" * 70)
print("HASIL EKSPERIMEN")
print("=" * 70)

display(result_df)

print()

print("=" * 70)
print("STATISTIK DESKRIPTIF")
print("=" * 70)

display(result_df.describe())

print("\nRata-rata Accuracy")

print(f"Naive Bayes : {np.mean(nb_accuracy):.4f}")

print(f"SVM         : {np.mean(svm_accuracy):.4f}")

print("\nStandar Deviasi")

print(f"Naive Bayes : {np.std(nb_accuracy):.4f}")

print(f"SVM         : {np.std(svm_accuracy):.4f}")

HASIL EKSPERIMEN


,Random State,Naive Bayes,SVM,Difference
0,42,0.94350,0.94950,0.00600
1,43,0.94075,0.94925,0.00850
2,44,0.94500,0.94950,0.00450
3,45,0.94900,0.95000,0.00100
4,46,0.94100,0.94150,0.00050
5,47,0.93950,0.94550,0.00600
6,48,0.94450,0.94700,0.00250
7,49,0.94750,0.95175,0.00425
8,50,0.94475,0.94725,0.00250
9,51,0.94950,0.95125,0.00175



STATISTIK DESKRIPTIF


,Random State,Naive Bayes,SVM,Difference
count,30.000000,30.000000,30.000000,30.000000
mean,56.500000,0.945458,0.948558,0.003100
std,8.803408,0.003833,0.003854,0.002755
min,42.000000,0.937500,0.940500,-0.002000
25%,49.250000,0.942875,0.946125,0.001187
50%,56.500000,0.945000,0.949250,0.002500
75%,63.750000,0.948187,0.950375,0.004875
max,71.000000,0.953750,0.959000,0.008500



Rata-rata Accuracy
Naive Bayes : 0.9455
SVM         : 0.9486

Standar Deviasi
Naive Bayes : 0.0038
SVM         : 0.0038


**TAHAP 10 :** PAIRED STUDENT'S T-TEST

In [17]:
ALPHA = 0.05

t_stat, t_pvalue = ttest_rel(

    nb_accuracy,

    svm_accuracy

)

mean_difference = np.mean(

    np.array(svm_accuracy)

    -

    np.array(nb_accuracy)

)

print("=" * 70)
print("PAIRED STUDENT'S T-TEST")
print("=" * 70)

print("\nHipotesis")

print("H0 : Tidak terdapat perbedaan performa yang signifikan.")

print("H1 : Terdapat perbedaan performa yang signifikan.")

print(f"\nAlpha       : {ALPHA}")

print(f"T-Statistic : {t_stat:.6f}")

print(f"P-Value     : {t_pvalue:.6f}")

print(f"Mean Diff   : {mean_difference:.6f}")

print()

if t_pvalue < ALPHA:

    print("Keputusan : Tolak H0")

    print("Kesimpulan : Perbedaan kedua model signifikan secara statistik.")

else:

    print("Keputusan : Gagal menolak H0")

    print("Kesimpulan : Tidak terdapat perbedaan yang signifikan secara statistik.")

PAIRED STUDENT'S T-TEST

Hipotesis
H0 : Tidak terdapat perbedaan performa yang signifikan.
H1 : Terdapat perbedaan performa yang signifikan.

Alpha       : 0.05
T-Statistic : -6.163622
P-Value     : 0.000001
Mean Diff   : 0.003100

Keputusan : Tolak H0
Kesimpulan : Perbedaan kedua model signifikan secara statistik.


**TAHAP 11 :** WILCOXON SIGNED-RANK TEST

In [18]:
ALPHA = 0.05

w_stat, w_pvalue = wilcoxon(

    nb_accuracy,

    svm_accuracy

)

median_difference = np.median(

    np.array(svm_accuracy)

    -

    np.array(nb_accuracy)

)

print("=" * 70)
print("WILCOXON SIGNED-RANK TEST")
print("=" * 70)

print("\nHipotesis")

print("H0 : Tidak terdapat perbedaan performa yang signifikan.")

print("H1 : Terdapat perbedaan performa yang signifikan.")

print(f"\nAlpha       : {ALPHA}")

print(f"W-Statistic : {w_stat:.6f}")

print(f"P-Value     : {w_pvalue:.6f}")

print(f"Median Diff : {median_difference:.6f}")

print()

if w_pvalue < ALPHA:

    print("Keputusan : Tolak H0")

    print("Kesimpulan : Perbedaan kedua model signifikan secara statistik.")

else:

    print("Keputusan : Gagal menolak H0")

    print("Kesimpulan : Tidak terdapat perbedaan yang signifikan secara statistik.")

WILCOXON SIGNED-RANK TEST

Hipotesis
H0 : Tidak terdapat perbedaan performa yang signifikan.
H1 : Terdapat perbedaan performa yang signifikan.

Alpha       : 0.05
W-Statistic : 17.000000
P-Value     : 0.000014
Median Diff : 0.002500

Keputusan : Tolak H0
Kesimpulan : Perbedaan kedua model signifikan secara statistik.


**TAHAP 12 :** INTERPRETASI HASIL UJI STATISTIK

In [19]:
print("=" * 70)
print("INTERPRETASI HASIL UJI STATISTIK")
print("=" * 70)

print(f"Rata-rata Accuracy Naive Bayes : {np.mean(nb_accuracy):.4f}")

print(f"Rata-rata Accuracy SVM         : {np.mean(svm_accuracy):.4f}")

print(f"Selisih Rata-rata              : {mean_difference:.4f}")

print()

print(f"T-Test     : t = {t_stat:.4f} | p = {t_pvalue:.6f}")

print(f"Wilcoxon   : W = {w_stat:.4f} | p = {w_pvalue:.6f}")

print()

if t_pvalue < ALPHA and w_pvalue < ALPHA:

    print("Kesimpulan:")

    print("- Kedua uji statistik menunjukkan adanya perbedaan performa yang signifikan.")

    print("- Support Vector Machine memiliki rata-rata accuracy yang lebih tinggi dibandingkan Naive Bayes.")

    print("- Perbedaan tersebut signifikan secara statistik pada tingkat signifikansi α = 0.05.")

else:

    print("Kesimpulan:")

    print("- Kedua uji statistik tidak menunjukkan perbedaan performa yang signifikan.")

INTERPRETASI HASIL UJI STATISTIK
Rata-rata Accuracy Naive Bayes : 0.9455
Rata-rata Accuracy SVM         : 0.9486
Selisih Rata-rata              : 0.0031

T-Test     : t = -6.1636 | p = 0.000001
Wilcoxon   : W = 17.0000 | p = 0.000014

Kesimpulan:
- Kedua uji statistik menunjukkan adanya perbedaan performa yang signifikan.
- Support Vector Machine memiliki rata-rata accuracy yang lebih tinggi dibandingkan Naive Bayes.
- Perbedaan tersebut signifikan secara statistik pada tingkat signifikansi α = 0.05.


**TAHAP 13 :** MEMBANGUN FUNGSI LSTM

In [20]:
import tensorflow as tf
import numpy as np
import random

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

print("=" * 70)
print("MEMBANGUN FUNGSI LSTM")
print("=" * 70)

#Parameter LSTM

MAX_WORDS = 5000
MAX_LENGTH = 20

EMBEDDING_DIM = 64
LSTM_UNITS = 64

EPOCHS = 20
BATCH_SIZE = 64

print("\nKonfigurasi Model")
print("-----------------------------")
print("Vocabulary :", MAX_WORDS)
print("Max Length :", MAX_LENGTH)
print("Embedding  :", EMBEDDING_DIM)
print("LSTM Unit  :", LSTM_UNITS)
print("Epoch      :", EPOCHS)
print("Batch Size :", BATCH_SIZE)

#Fungsi Train LSTM

def train_lstm(random_state=SEED):

    #Random Seed

    random.seed(random_state)

    np.random.seed(random_state)

    tf.random.set_seed(random_state)

    try:

        tf.keras.utils.set_random_seed(random_state)

    except:

        pass

    tf.keras.backend.clear_session()

    #Split Dataset

    X_train, X_test, y_train, y_test = split_data(random_state)

    #Tokenizer

    lstm_tokenizer = Tokenizer(

        num_words=MAX_WORDS,

        oov_token="<oov>"

    )

    lstm_tokenizer.fit_on_texts(X_train)

    X_train_seq = lstm_tokenizer.texts_to_sequences(
        X_train
    )

    X_test_seq = lstm_tokenizer.texts_to_sequences(
        X_test
    )

    X_train_pad = pad_sequences(

        X_train_seq,

        maxlen=MAX_LENGTH,

        padding="post"

    )

    X_test_pad = pad_sequences(

        X_test_seq,

        maxlen=MAX_LENGTH,

        padding="post"

    )

    #One Hot Encoding

    y_train_cat = to_categorical(

        y_train,

        num_classes=3

    )

    y_test_cat = to_categorical(

        y_test,

        num_classes=3

    )

    #Model LSTM

    lstm_model = Sequential()

    lstm_model.add(

        Embedding(

            input_dim=MAX_WORDS,

            output_dim=EMBEDDING_DIM,

            input_length=MAX_LENGTH

        )

    )

    lstm_model.add(

        LSTM(

            LSTM_UNITS

        )

    )

    lstm_model.add(

        Dense(

            3,

            activation="softmax"

        )

    )

    #Compile

    lstm_model.compile(

        optimizer="adam",

        loss="categorical_crossentropy",

        metrics=[

            "accuracy"

        ]

    )

    #Early Stopping

    early_stop = EarlyStopping(

        monitor="val_loss",

        patience=3,

        restore_best_weights=True

    )

    #Training

    history = lstm_model.fit(

        X_train_pad,

        y_train_cat,

        validation_split=0.10,

        epochs=EPOCHS,

        batch_size=BATCH_SIZE,

        callbacks=[early_stop],

        verbose=0

    )

    #Prediction

    lstm_probability = lstm_model.predict(

        X_test_pad,

        verbose=0

    )

    lstm_pred = np.argmax(

        lstm_probability,

        axis=1

    )

    #Evaluation

    acc = accuracy_score(

        y_test,

        lstm_pred

    )

    pre = precision_score(

        y_test,

        lstm_pred,

        average="weighted"

    )

    rec = recall_score(

        y_test,

        lstm_pred,

        average="weighted"

    )

    f1 = f1_score(

        y_test,

        lstm_pred,

        average="weighted"

    )

    return {

        "accuracy": acc,

        "precision": pre,

        "recall": rec,

        "f1": f1,

        "X_test_lstm": list(X_test),

        "y_test_lstm": y_test,

        "lstm_pred": lstm_pred,

        "history": history.history,

        "tokenizer": lstm_tokenizer,

        "model": lstm_model

    }

print("\n")
print("=" * 70)
print("Fungsi train_lstm() berhasil dibuat.")
print("=" * 70)

MEMBANGUN FUNGSI LSTM

Konfigurasi Model
-----------------------------
Vocabulary : 5000
Max Length : 20
Embedding  : 64
LSTM Unit  : 64
Epoch      : 20
Batch Size : 64


Fungsi train_lstm() berhasil dibuat.


**TAHAP 14 :** EKSPERIMEN LSTM

In [22]:
random_states = list(range(SEED, SEED + 30))

lstm_accuracy = []
lstm_precision = []
lstm_recall = []
lstm_f1 = []

print("=" * 70)
print("EKSPERIMEN LONG SHORT-TERM MEMORY")
print("=" * 70)

for rs in random_states:

    lstm_output = train_lstm(rs)

    lstm_accuracy.append(

        lstm_output["accuracy"]

    )

    lstm_precision.append(

        lstm_output["precision"]

    )

    lstm_recall.append(

        lstm_output["recall"]

    )

    lstm_f1.append(

        lstm_output["f1"]

    )

    print(

        f"Seed {rs} | "

        f"Accuracy = {lstm_output['accuracy']:.4f}"

    )

#Hasil Eksperimen

lstm_result = pd.DataFrame({

    "Seed": random_states,

    "Accuracy": lstm_accuracy,

    "Precision": lstm_precision,

    "Recall": lstm_recall,

    "F1-score": lstm_f1

})

print("\n")

print("=" * 70)
print("RATA-RATA LONG SHORT-TERM MEMORY")
print("=" * 70)

print(f"Accuracy      : {np.mean(lstm_accuracy):.4f}")
print(f"Precision     : {np.mean(lstm_precision):.4f}")
print(f"Recall        : {np.mean(lstm_recall):.4f}")
print(f"F1-score      : {np.mean(lstm_f1):.4f}")
print(f"Std Accuracy  : {np.std(lstm_accuracy):.4f}")

print("\n5 Hasil Pertama")

display(lstm_result.head())

EKSPERIMEN LONG SHORT-TERM MEMORY
Seed 42 | Accuracy = 0.9487
Seed 43 | Accuracy = 0.9527
Seed 44 | Accuracy = 0.9525
Seed 45 | Accuracy = 0.9515
Seed 46 | Accuracy = 0.9460
Seed 47 | Accuracy = 0.9480
Seed 48 | Accuracy = 0.9515
Seed 49 | Accuracy = 0.9575
Seed 50 | Accuracy = 0.9507
Seed 51 | Accuracy = 0.9500
Seed 52 | Accuracy = 0.9437
Seed 53 | Accuracy = 0.9560
Seed 54 | Accuracy = 0.9575
Seed 55 | Accuracy = 0.9475
Seed 56 | Accuracy = 0.9397
Seed 57 | Accuracy = 0.9513
Seed 58 | Accuracy = 0.9507
Seed 59 | Accuracy = 0.9457
Seed 60 | Accuracy = 0.9425
Seed 61 | Accuracy = 0.9513
Seed 62 | Accuracy = 0.9457
Seed 63 | Accuracy = 0.9517
Seed 64 | Accuracy = 0.9503
Seed 65 | Accuracy = 0.9523
Seed 66 | Accuracy = 0.9495
Seed 67 | Accuracy = 0.9530
Seed 68 | Accuracy = 0.9535
Seed 69 | Accuracy = 0.9480
Seed 70 | Accuracy = 0.9547
Seed 71 | Accuracy = 0.9520


RATA-RATA LONG SHORT-TERM MEMORY
Accuracy      : 0.9502
Precision     : 0.9558
Recall        : 0.9502
F1-score      : 0.9497

,Seed,Accuracy,Precision,Recall,F1-score
0,42,0.94875,0.955296,0.94875,0.948105
1,43,0.95275,0.958370,0.95275,0.952224
2,44,0.95250,0.958176,0.95250,0.951967
3,45,0.95150,0.957403,0.95150,0.950939
4,46,0.94600,0.953218,0.94600,0.945263


**TAHAP 15 :** REKAPITULASI HASIL EKSPERIMEN

In [23]:
experiment_summary = pd.DataFrame({

    "Random State": random_states,

    "Naive Bayes": nb_accuracy,

    "SVM": svm_accuracy,

    "LSTM": lstm_accuracy

})

print("=" * 70)
print("REKAPITULASI HASIL EKSPERIMEN")
print("=" * 70)

display(experiment_summary)

print("\n")

print("=" * 70)
print("STATISTIK DESKRIPTIF")
print("=" * 70)

display(

    experiment_summary.describe()

)

print("\n")

print("=" * 70)
print("RATA-RATA SETIAP MODEL")
print("=" * 70)

print(f"Naive Bayes : {experiment_summary['Naive Bayes'].mean():.4f}")
print(f"SVM         : {experiment_summary['SVM'].mean():.4f}")
print(f"LSTM        : {experiment_summary['LSTM'].mean():.4f}")

print("\nStandar Deviasi")

print(f"Naive Bayes : {experiment_summary['Naive Bayes'].std():.4f}")
print(f"SVM         : {experiment_summary['SVM'].std():.4f}")
print(f"LSTM        : {experiment_summary['LSTM'].std():.4f}")

print("\n")

print("=" * 70)
print("RANKING MODEL")
print("=" * 70)

ranking = (

    experiment_summary[

        ["Naive Bayes", "SVM", "LSTM"]

    ]

    .mean()

    .sort_values(

        ascending=False

    )

)

for i, (model, nilai) in enumerate(

    ranking.items(),

    start=1

):

    print(f"{i}. {model:<15} : {nilai:.4f}")

best_model = ranking.index[0]

best_accuracy = ranking.iloc[0]

print("\nModel Terbaik")

print(f"{best_model} ({best_accuracy:.4f})")

output_file = "statistical_summary.csv"

experiment_summary.to_csv(

    output_file,

    index=False

)

print("\n")

print(f"{output_file} berhasil disimpan.")

REKAPITULASI HASIL EKSPERIMEN


,Random State,Naive Bayes,SVM,LSTM
0,42,0.94350,0.94950,0.94875
1,43,0.94075,0.94925,0.95275
2,44,0.94500,0.94950,0.95250
3,45,0.94900,0.95000,0.95150
4,46,0.94100,0.94150,0.94600
5,47,0.93950,0.94550,0.94800
6,48,0.94450,0.94700,0.95150
7,49,0.94750,0.95175,0.95750
8,50,0.94475,0.94725,0.95075
9,51,0.94950,0.95125,0.95000




STATISTIK DESKRIPTIF


,Random State,Naive Bayes,SVM,LSTM
count,30.000000,30.000000,30.000000,30.000000
mean,56.500000,0.945458,0.948558,0.950200
std,8.803408,0.003833,0.003854,0.004129
min,42.000000,0.937500,0.940500,0.939750
25%,49.250000,0.942875,0.946125,0.948000
50%,56.500000,0.945000,0.949250,0.951000
75%,63.750000,0.948187,0.950375,0.952438
max,71.000000,0.953750,0.959000,0.957500




RATA-RATA SETIAP MODEL
Naive Bayes : 0.9455
SVM         : 0.9486
LSTM        : 0.9502

Standar Deviasi
Naive Bayes : 0.0038
SVM         : 0.0039
LSTM        : 0.0041


RANKING MODEL
1. LSTM            : 0.9502
2. SVM             : 0.9486
3. Naive Bayes     : 0.9455

Model Terbaik
LSTM (0.9502)


statistical_summary.csv berhasil disimpan.


T**AHAP 16 :** PAIRED STUDENT'S T-TEST

In [24]:
print("=" * 70)
print("PAIRED STUDENT'S T-TEST")
print("=" * 70)

hasil_ttest = []

pasangan = [

    ("Naive Bayes", "SVM"),

    ("Naive Bayes", "LSTM"),

    ("SVM", "LSTM")

]

for model1, model2 in pasangan:

    t_stat, p_value = ttest_rel(

        experiment_summary[model1],

        experiment_summary[model2]

    )

    mean_difference = (

        experiment_summary[model2].mean()

        -

        experiment_summary[model1].mean()

    )

    keputusan = (

        "Signifikan"

        if p_value < ALPHA

        else

        "Tidak Signifikan"

    )

    hasil_ttest.append({

        "Perbandingan": f"{model1} vs {model2}",

        "Mean Difference": round(mean_difference, 4),

        "T-Statistic": round(t_stat, 4),

        "P-Value": round(p_value, 6),

        "Kesimpulan": keputusan

    })

ttest_df = pd.DataFrame(

    hasil_ttest

)

display(

    ttest_df

)

print("\n")

print("=" * 70)
print("INTERPRETASI")
print("=" * 70)

for _, row in ttest_df.iterrows():

    print(

        f"{row['Perbandingan']}"

        f" -> "

        f"{row['Kesimpulan']}"

    )

output_file = "paired_ttest_result.csv"

ttest_df.to_csv(

    output_file,

    index=False

)

print("\n")

print(f"{output_file} berhasil disimpan.")

PAIRED STUDENT'S T-TEST


,Perbandingan,Mean Difference,T-Statistic,P-Value,Kesimpulan
0,Naive Bayes vs SVM,0.0031,-6.1636,0.000001,Signifikan
1,Naive Bayes vs LSTM,0.0047,-4.8974,0.000034,Signifikan
2,SVM vs LSTM,0.0016,-1.9262,0.063933,Tidak Signifikan




INTERPRETASI
Naive Bayes vs SVM -> Signifikan
Naive Bayes vs LSTM -> Signifikan
SVM vs LSTM -> Tidak Signifikan


paired_ttest_result.csv berhasil disimpan.


**TAHAP 17 :** WILCOXON SIGNED-RANK TEST

In [25]:
print("=" * 70)
print("WILCOXON SIGNED-RANK TEST")
print("=" * 70)

hasil_wilcoxon = []

pasangan = [

    ("Naive Bayes", "SVM"),

    ("Naive Bayes", "LSTM"),

    ("SVM", "LSTM")

]

for model1, model2 in pasangan:

    w_stat, p_value = wilcoxon(

        experiment_summary[model1],

        experiment_summary[model2]

    )

    median_difference = np.median(

        experiment_summary[model2]

        -

        experiment_summary[model1]

    )

    keputusan = (

        "Signifikan"

        if p_value < ALPHA

        else

        "Tidak Signifikan"

    )

    hasil_wilcoxon.append({

        "Perbandingan": f"{model1} vs {model2}",

        "Median Difference": round(median_difference, 4),

        "W-Statistic": round(w_stat, 4),

        "P-Value": round(p_value, 6),

        "Kesimpulan": keputusan

    })

wilcoxon_df = pd.DataFrame(

    hasil_wilcoxon

)

display(

    wilcoxon_df

)

print("\n")

print("=" * 70)
print("INTERPRETASI")
print("=" * 70)

for _, row in wilcoxon_df.iterrows():

    print(

        f"{row['Perbandingan']}"

        f" -> "

        f"{row['Kesimpulan']}"

    )

output_file = "wilcoxon_result.csv"

wilcoxon_df.to_csv(

    output_file,

    index=False

)

print("\n")

print(f"{output_file} berhasil disimpan.")

WILCOXON SIGNED-RANK TEST


,Perbandingan,Median Difference,W-Statistic,P-Value,Kesimpulan
0,Naive Bayes vs SVM,0.0025,17.0,0.000014,Signifikan
1,Naive Bayes vs LSTM,0.0059,53.0,0.000222,Signifikan
2,SVM vs LSTM,0.0027,124.5,0.026305,Signifikan




INTERPRETASI
Naive Bayes vs SVM -> Signifikan
Naive Bayes vs LSTM -> Signifikan
SVM vs LSTM -> Signifikan


wilcoxon_result.csv berhasil disimpan.


**TAHAP 18 :** MENYIAPKAN PREDIKSI BASELINE (SEED 42)

In [26]:
print("=" * 70)
print("MENYIAPKAN PREDIKSI BASELINE (SEED 42)")
print("=" * 70)

#Split Data

X_train, X_test, y_train, y_test = split_data(SEED)

#Naive Bayes

nb_model = Pipeline([

    (

        "tfidf",

        TfidfVectorizer(

            max_features=5000,

            ngram_range=(1,2)

        )

    ),

    (

        "model",

        MultinomialNB()

    )

])

nb_model.fit(

    X_train,

    y_train

)

nb_pred = nb_model.predict(

    X_test

)

#Support Vector Machine

svm_model = Pipeline([

    (

        "tfidf",

        TfidfVectorizer(

            max_features=5000,

            ngram_range=(1,2)

        )

    ),

    (

        "model",

        LinearSVC(

            random_state=SEED

        )

    )

])

svm_model.fit(

    X_train,

    y_train

)

svm_pred = svm_model.predict(

    X_test

)

#Long Short-Term Memory

hasil_lstm = train_lstm(SEED)

X_test_lstm = hasil_lstm["X_test_lstm"]

y_test_lstm = hasil_lstm["y_test_lstm"]

lstm_pred = hasil_lstm["lstm_pred"]

print("\n")

print("=" * 70)
print("DATA BERHASIL DISIAPKAN")
print("=" * 70)

print(f"Jumlah Testing NB/SVM : {len(y_test)}")

print(f"Jumlah Testing LSTM   : {len(y_test_lstm)}")

print("\nVariabel tersedia")

print("- X_test")

print("- y_test")

print("- X_test_lstm")

print("- y_test_lstm")

print("- nb_pred")

print("- svm_pred")

print("- lstm_pred")

MENYIAPKAN PREDIKSI BASELINE (SEED 42)


DATA BERHASIL DISIAPKAN
Jumlah Testing NB/SVM : 4000
Jumlah Testing LSTM   : 4000

Variabel tersedia
- X_test
- y_test
- X_test_lstm
- y_test_lstm
- nb_pred
- svm_pred
- lstm_pred


In [28]:
assert len(y_test) == len(y_test_lstm)

assert np.array_equal(
    np.array(y_test),
    np.array(y_test_lstm)
), "Ground truth NB/SVM dan LSTM berbeda."

**TAHAP 19 :** MCNEMAR TEST

In [29]:
from statsmodels.stats.contingency_tables import mcnemar

print("=" * 70)
print("MCNEMAR TEST")
print("=" * 70)

def run_mcnemar(

    model1_name,

    pred1,

    model2_name,

    pred2,

    y_true

):

    correct1 = pred1 == y_true

    correct2 = pred2 == y_true

    b = ((correct1 == True) & (correct2 == False)).sum()

    c = ((correct1 == False) & (correct2 == True)).sum()

    contingency_table = [

        [0, b],

        [c, 0]

    ]

    result = mcnemar(

        contingency_table,

        exact=False,

        correction=True

    )

    keputusan = (

        "Signifikan"

        if result.pvalue < ALPHA

        else

        "Tidak Signifikan"

    )

    return {

        "Perbandingan": f"{model1_name} vs {model2_name}",

        "B": b,

        "C": c,

        "B + C": b + c,

        "Statistic": round(result.statistic, 4),

        "P-Value": round(result.pvalue, 6),

        "Kesimpulan": keputusan

    }

hasil_mcnemar = []

hasil_mcnemar.append(

    run_mcnemar(

        "Naive Bayes",

        nb_pred,

        "SVM",

        svm_pred,

        y_test

    )

)

hasil_mcnemar.append(

    run_mcnemar(

        "Naive Bayes",

        nb_pred,

        "LSTM",

        lstm_pred,

        y_test_lstm

    )

)

hasil_mcnemar.append(

    run_mcnemar(

        "SVM",

        svm_pred,

        "LSTM",

        lstm_pred,

        y_test_lstm

    )

)

mcnemar_df = pd.DataFrame(

    hasil_mcnemar

)

display(

    mcnemar_df

)

print("\n")

print("=" * 70)
print("INTERPRETASI")
print("=" * 70)

for _, row in mcnemar_df.iterrows():

    print(

        f"{row['Perbandingan']}"

        f" -> "

        f"{row['Kesimpulan']}"

    )

output_file = "mcnemar_result.csv"

mcnemar_df.to_csv(

    output_file,

    index=False

)

print("\n")

print(f"{output_file} berhasil disimpan.")

MCNEMAR TEST


,Perbandingan,B,C,B + C,Statistic,P-Value,Kesimpulan
0,Naive Bayes vs SVM,71,95,166,3.1867,0.074238,Tidak Signifikan
1,Naive Bayes vs LSTM,82,103,185,2.1622,0.141446,Tidak Signifikan
2,SVM vs LSTM,72,69,141,0.0284,0.866245,Tidak Signifikan




INTERPRETASI
Naive Bayes vs SVM -> Tidak Signifikan
Naive Bayes vs LSTM -> Tidak Signifikan
SVM vs LSTM -> Tidak Signifikan


mcnemar_result.csv berhasil disimpan.


**TAHAP 20 :** MEMBANDINGKAN HASIL PREDIKSI SETIAP MODEL

In [30]:
print("=" * 70)
print("MEMBANGUN PREDICTION COMPARISON")
print("=" * 70)

#Validasi Data Testing

assert np.array_equal(

    np.array(y_test),

    np.array(y_test_lstm)

), "Ground truth berbeda."

assert list(X_test) == list(X_test_lstm), "Data testing berbeda."

#Dataframe Seluruh Prediksi

comparison_df = pd.DataFrame({

    "Text": list(X_test),

    "Actual Label": label_encoder.inverse_transform(y_test),

    "Naive Bayes": label_encoder.inverse_transform(nb_pred),

    "SVM": label_encoder.inverse_transform(svm_pred),

    "LSTM": label_encoder.inverse_transform(lstm_pred)

})

comparison_df["Agreement"] = (

    (comparison_df["Naive Bayes"] == comparison_df["SVM"])

    &

    (comparison_df["SVM"] == comparison_df["LSTM"])

)

#Prediksi Berbeda

comparison_diff = comparison_df[

    ~comparison_df["Agreement"]

].copy()

print("\n")

print("=" * 70)
print("CONTOH DATA DENGAN PREDIKSI BERBEDA")
print("=" * 70)

display(

    comparison_diff.head(20)

)

#Simpan CSV

comparison_file = "prediction_comparison.csv"

difference_file = "prediction_difference.csv"

comparison_df.to_csv(

    comparison_file,

    index=False

)

comparison_diff.to_csv(

    difference_file,

    index=False

)

#Ringkasan

jumlah_data = len(comparison_df)

jumlah_sepakat = comparison_df["Agreement"].sum()

jumlah_berbeda = len(comparison_diff)

print("\n")

print("=" * 70)
print("RINGKASAN")
print("=" * 70)

print(f"Jumlah Data Testing  : {jumlah_data}")

print(f"Semua Model Sepakat  : {jumlah_sepakat}")

print(f"Prediksi Berbeda     : {jumlah_berbeda}")

print(f"Persentase Perbedaan : {(jumlah_berbeda/jumlah_data)*100:.2f}%")

print("\n")

print("=" * 70)
print("FILE BERHASIL DISIMPAN")
print("=" * 70)

print(f"- {comparison_file}")

print(f"- {difference_file}")

MEMBANGUN PREDICTION COMPARISON


CONTOH DATA DENGAN PREDIKSI BERBEDA


,Text,Actual Label,Naive Bayes,SVM,LSTM,Agreement
12,warnanya lumayan,Netral,Netral,Netral,Positif,False
14,modelnya estetik jujur,Netral,Positif,Netral,Positif,False
27,kombinasi warnanya lumayan sih,Positif,Netral,Positif,Positif,False
56,outfitnya unik haha,Positif,Netral,Positif,Positif,False
59,warnanya oke menurutku,Netral,Netral,Netral,Positif,False
60,desainnya unik sih,Positif,Netral,Positif,Positif,False
97,looknya menarik wkwk,Netral,Positif,Netral,Positif,False
114,looknya menarik wkwk,Netral,Positif,Netral,Positif,False
159,konsepnya oke haha,Netral,Netral,Netral,Positif,False
160,kombinasi warnanya estetik wkwk,Netral,Positif,Netral,Positif,False




RINGKASAN
Jumlah Data Testing  : 4000
Semua Model Sepakat  : 3754
Prediksi Berbeda     : 246
Persentase Perbedaan : 6.15%


FILE BERHASIL DISIMPAN
- prediction_comparison.csv
- prediction_difference.csv


**TAHAP 21 :** ERROR ANALYSIS

In [31]:
print("=" * 70)
print("ERROR ANALYSIS")
print("=" * 70)

#Salin Data

error_df = comparison_diff.copy()

#Cek Prediksi Benar

error_df["NB Correct"] = (

    error_df["Naive Bayes"]

    ==

    error_df["Actual Label"]

)

error_df["SVM Correct"] = (

    error_df["SVM"]

    ==

    error_df["Actual Label"]

)

error_df["LSTM Correct"] = (

    error_df["LSTM"]

    ==

    error_df["Actual Label"]

)

#Jumlah Model Benar

error_df["Jumlah Model Benar"] = (

    error_df["NB Correct"].astype(int)

    +

    error_df["SVM Correct"].astype(int)

    +

    error_df["LSTM Correct"].astype(int)

)

#Model yang Benar

def best_model(row):

    models = []

    if row["NB Correct"]:

        models.append("Naive Bayes")

    if row["SVM Correct"]:

        models.append("SVM")

    if row["LSTM Correct"]:

        models.append("LSTM")

    if len(models) == 0:

        return "Tidak Ada"

    return " + ".join(models)

error_df["Best Model"] = error_df.apply(

    best_model,

    axis=1

)

#Simpan CSV

output_file = "error_analysis.csv"

error_df.to_csv(

    output_file,

    index=False

)

print("\n")

print("=" * 70)
print("RINGKASAN ERROR ANALYSIS")
print("=" * 70)

print(f"Jumlah data dianalisis : {len(error_df)}")

print()

print(f"NB benar   : {error_df['NB Correct'].sum()}")

print(f"SVM benar  : {error_df['SVM Correct'].sum()}")

print(f"LSTM benar : {error_df['LSTM Correct'].sum()}")

print("\nDistribusi Jumlah Model yang Benar")

print(

    error_df["Jumlah Model Benar"]

    .value_counts()

    .sort_index()

)

print("\nDistribusi Model Terbaik")

print(

    error_df["Best Model"]

    .value_counts()

)

print("\n")

print("=" * 70)
print("CONTOH ERROR ANALYSIS")
print("=" * 70)

display(

    error_df.head(20)

)

print("\n")

print(f"{output_file} berhasil disimpan.")

ERROR ANALYSIS


RINGKASAN ERROR ANALYSIS
Jumlah data dianalisis : 246

NB benar   : 112
SVM benar  : 136
LSTM benar : 133

Distribusi Jumlah Model yang Benar
Jumlah Model Benar
1    111
2    135
Name: count, dtype: int64

Distribusi Model Terbaik
Best Model
SVM + LSTM            64
Naive Bayes + SVM     41
Naive Bayes           41
LSTM                  39
SVM                   31
Naive Bayes + LSTM    30
Name: count, dtype: int64


CONTOH ERROR ANALYSIS


,Text,Actual Label,Naive Bayes,SVM,LSTM,Agreement,NB Correct,SVM Correct,LSTM Correct,Jumlah Model Benar,Best Model
12,warnanya lumayan,Netral,Netral,Netral,Positif,False,True,True,False,2,Naive Bayes + SVM
14,modelnya estetik jujur,Netral,Positif,Netral,Positif,False,False,True,False,1,SVM
27,kombinasi warnanya lumayan sih,Positif,Netral,Positif,Positif,False,False,True,True,2,SVM + LSTM
56,outfitnya unik haha,Positif,Netral,Positif,Positif,False,False,True,True,2,SVM + LSTM
59,warnanya oke menurutku,Netral,Netral,Netral,Positif,False,True,True,False,2,Naive Bayes + SVM
60,desainnya unik sih,Positif,Netral,Positif,Positif,False,False,True,True,2,SVM + LSTM
97,looknya menarik wkwk,Netral,Positif,Netral,Positif,False,False,True,False,1,SVM
114,looknya menarik wkwk,Netral,Positif,Netral,Positif,False,False,True,False,1,SVM
159,konsepnya oke haha,Netral,Netral,Netral,Positif,False,True,True,False,2,Naive Bayes + SVM
160,kombinasi warnanya estetik wkwk,Netral,Positif,Netral,Positif,False,False,True,False,1,SVM




error_analysis.csv berhasil disimpan.


**TAHAP 22 :** KATEGORI ERROR ANTAR MODEL

In [32]:
print("=" * 70)
print("KATEGORI ERROR ANTAR MODEL")
print("=" * 70)

#Fungsi Kategori

def kategori_error(row):

    nb = row["NB Correct"]
    svm = row["SVM Correct"]
    lstm = row["LSTM Correct"]

    if nb and not svm and not lstm:
        return "Only NB Correct"

    elif svm and not nb and not lstm:
        return "Only SVM Correct"

    elif lstm and not nb and not svm:
        return "Only LSTM Correct"

    elif nb and svm and not lstm:
        return "NB + SVM Correct"

    elif nb and lstm and not svm:
        return "NB + LSTM Correct"

    elif svm and lstm and not nb:
        return "SVM + LSTM Correct"

    elif nb and svm and lstm:
        return "All Correct"

    else:
        return "All Wrong"

#Tambah Kolom

error_df["Error Category"] = error_df.apply(

    kategori_error,

    axis=1

)

#Urutan Kategori

category_order = [

    "Only NB Correct",

    "Only SVM Correct",

    "Only LSTM Correct",

    "NB + SVM Correct",

    "NB + LSTM Correct",

    "SVM + LSTM Correct",

    "All Correct",

    "All Wrong"

]

error_df["Error Category"] = pd.Categorical(

    error_df["Error Category"],

    categories=category_order,

    ordered=True

)

#Ringkasan

kategori_summary = (

    error_df["Error Category"]

    .value_counts()

    .sort_values(ascending=False)

    .reset_index()

)

kategori_summary.columns = [

    "Error Category",

    "Jumlah"

]

kategori_summary["Persentase"] = (

    kategori_summary["Jumlah"]

    / len(error_df)

    * 100

).round(2)

print("\n")

print("=" * 70)
print("RINGKASAN KATEGORI")
print("=" * 70)

display(kategori_summary)

top_category = kategori_summary.iloc[0]

print("\nKategori Dominan")

print(

    f"{top_category['Error Category']} "

    f"({top_category['Jumlah']} data)"

)

#Simpan CSV

summary_file = "error_category_summary.csv"

detail_file = "error_analysis_category.csv"

kategori_summary.to_csv(

    summary_file,

    index=False

)

error_df.to_csv(

    detail_file,

    index=False

)

print("\n")

print("=" * 70)
print("CONTOH DATA")
print("=" * 70)

display(

    error_df[

        [

            "Text",

            "Actual Label",

            "Naive Bayes",

            "SVM",

            "LSTM",

            "Error Category"

        ]

    ].head(20)

)

print("\n")

print(f"{summary_file} berhasil disimpan.")

print(f"{detail_file} berhasil disimpan.")

KATEGORI ERROR ANTAR MODEL


RINGKASAN KATEGORI


,Error Category,Jumlah,Persentase
0,SVM + LSTM Correct,64,26.02
1,Only NB Correct,41,16.67
2,NB + SVM Correct,41,16.67
3,Only LSTM Correct,39,15.85
4,Only SVM Correct,31,12.60
5,NB + LSTM Correct,30,12.20
6,All Correct,0,0.00
7,All Wrong,0,0.00



Kategori Dominan
SVM + LSTM Correct (64 data)


CONTOH DATA


,Text,Actual Label,Naive Bayes,SVM,LSTM,Error Category
12,warnanya lumayan,Netral,Netral,Netral,Positif,NB + SVM Correct
14,modelnya estetik jujur,Netral,Positif,Netral,Positif,Only SVM Correct
27,kombinasi warnanya lumayan sih,Positif,Netral,Positif,Positif,SVM + LSTM Correct
56,outfitnya unik haha,Positif,Netral,Positif,Positif,SVM + LSTM Correct
59,warnanya oke menurutku,Netral,Netral,Netral,Positif,NB + SVM Correct
60,desainnya unik sih,Positif,Netral,Positif,Positif,SVM + LSTM Correct
97,looknya menarik wkwk,Netral,Positif,Netral,Positif,Only SVM Correct
114,looknya menarik wkwk,Netral,Positif,Netral,Positif,Only SVM Correct
159,konsepnya oke haha,Netral,Netral,Netral,Positif,NB + SVM Correct
160,kombinasi warnanya estetik wkwk,Netral,Positif,Netral,Positif,Only SVM Correct




error_category_summary.csv berhasil disimpan.
error_analysis_category.csv berhasil disimpan.


**TAHAP 23 :** TOP ERROR CASES

In [33]:
print("=" * 70)
print("TOP ERROR CASES")
print("=" * 70)

category_list = [

    "Only NB Correct",

    "Only SVM Correct",

    "Only LSTM Correct",

    "NB + SVM Correct",

    "NB + LSTM Correct",

    "SVM + LSTM Correct",

    "All Wrong"

]

top_cases = []

for category in category_list:

    subset = error_df[

        error_df["Error Category"] == category

    ].head(5)

    if not subset.empty:

        top_cases.append(subset)

top_cases_df = pd.concat(

    top_cases,

    ignore_index=True

)

top_cases_df["Error Category"] = pd.Categorical(

    top_cases_df["Error Category"],

    categories=category_list,

    ordered=True

)

top_cases_df = top_cases_df.sort_values(

    by="Error Category"

)

print("\n")

print("=" * 70)
print("CONTOH KASUS TERPILIH")
print("=" * 70)

display(

    top_cases_df[

        [

            "Text",

            "Actual Label",

            "Naive Bayes",

            "SVM",

            "LSTM",

            "Error Category"

        ]

    ]

)

output_file = "top_error_cases.csv"

top_cases_df.to_csv(

    output_file,

    index=False

)

print("\n")

print("=" * 70)
print("JUMLAH CONTOH PER KATEGORI")
print("=" * 70)

for category in category_list:

    jumlah = len(

        top_cases_df[

            top_cases_df["Error Category"] == category

        ]

    )

    print(f"{category:<22} : {jumlah}")

print("\n")

print(f"Total contoh : {len(top_cases_df)}")

print("\n")

print("=" * 70)
print("FILE BERHASIL DISIMPAN")
print("=" * 70)

print(f"- {output_file}")

TOP ERROR CASES


CONTOH KASUS TERPILIH


,Text,Actual Label,Naive Bayes,SVM,LSTM,Error Category
0,kombinasi warnanya oke haha,Netral,Netral,Positif,Positif,Only NB Correct
1,desainnya cukup bagus,Netral,Netral,Positif,Positif,Only NB Correct
2,modelnya oke jujur,Netral,Netral,Positif,Positif,Only NB Correct
3,warnanya unik,Netral,Netral,Positif,Positif,Only NB Correct
4,desainnya oke imo,Netral,Netral,Positif,Positif,Only NB Correct
5,modelnya estetik jujur,Netral,Positif,Netral,Positif,Only SVM Correct
6,looknya menarik wkwk,Netral,Positif,Netral,Positif,Only SVM Correct
7,looknya menarik wkwk,Netral,Positif,Netral,Positif,Only SVM Correct
8,kombinasi warnanya estetik wkwk,Netral,Positif,Netral,Positif,Only SVM Correct
9,modelnya estetik jujur,Netral,Positif,Netral,Positif,Only SVM Correct




JUMLAH CONTOH PER KATEGORI
Only NB Correct        : 5
Only SVM Correct       : 5
Only LSTM Correct      : 5
NB + SVM Correct       : 5
NB + LSTM Correct      : 5
SVM + LSTM Correct     : 5
All Wrong              : 0


Total contoh : 30


FILE BERHASIL DISIMPAN
- top_error_cases.csv


**TAHAP 24 :** CLASSIFICATION REPORT + FP/FN ANALYSIS

In [34]:
#Classification Report + FP/FN Analysis

print("=" * 70)
print("CLASSIFICATION REPORT + FP/FN ANALYSIS")
print("=" * 70)

label_names = [

    "Negatif",

    "Netral",

    "Positif"

]

models = {

    "Naive Bayes": {

        "pred": nb_pred,

        "y_true": y_test

    },

    "SVM": {

        "pred": svm_pred,

        "y_true": y_test

    },

    "LSTM": {

        "pred": lstm_pred,

        "y_true": y_test_lstm

    }

}

classification_summary = []

for model_name, model_data in models.items():

    pred = model_data["pred"]

    y_true = model_data["y_true"]

    print("\n")
    print("=" * 70)
    print(model_name.upper())
    print("=" * 70)

    #Classification Report

    report = classification_report(

        y_true,

        pred,

        target_names=label_names,

        output_dict=True

    )

    report_df = pd.DataFrame(report).transpose()

    display(report_df)

    output_prefix = model_name.lower().replace(" ", "_")

    report_df.to_csv(

        f"{output_prefix}_classification_report.csv",

        index=True

    )

    #Confusion Matrix

    cm = confusion_matrix(

        y_true,

        pred

    )

    cm_df = pd.DataFrame(

        cm,

        index=label_names,

        columns=label_names

    )

    print("\nConfusion Matrix")

    display(cm_df)

    cm_df.to_csv(

        f"{output_prefix}_confusion_matrix.csv"

    )

    #FP dan FN

    fp = cm.sum(axis=0) - np.diag(cm)

    fn = cm.sum(axis=1) - np.diag(cm)

    fpfn_df = pd.DataFrame({

        "Class": label_names,

        "False Positive": fp,

        "False Negative": fn

    })

    print("\nFP / FN")

    display(fpfn_df)

    fpfn_df.to_csv(

        f"{output_prefix}_fp_fn.csv",

        index=False

    )

    classification_summary.append({

        "Model": model_name,

        "Accuracy": report["accuracy"],

        "Macro Precision": report["macro avg"]["precision"],

        "Macro Recall": report["macro avg"]["recall"],

        "Macro F1": report["macro avg"]["f1-score"],

        "Weighted F1": report["weighted avg"]["f1-score"]

    })

#Ringkasan

summary_df = pd.DataFrame(

    classification_summary

)

print("\n")

print("=" * 70)
print("RINGKASAN PERBANDINGAN")
print("=" * 70)

display(summary_df)

summary_df.to_csv(

    "classification_summary.csv",

    index=False

)

print("\n")

print("=" * 70)
print("FILE BERHASIL DISIMPAN")
print("=" * 70)

print("- classification_summary.csv")

print("- *_classification_report.csv")

print("- *_confusion_matrix.csv")

print("- *_fp_fn.csv")

CLASSIFICATION REPORT + FP/FN ANALYSIS


NAIVE BAYES


,precision,recall,f1-score,support
Negatif,0.999286,1.000000,0.999643,1400.0000
Netral,0.913486,0.897500,0.905422,1200.0000
Positif,0.913380,0.926429,0.919858,1400.0000
accuracy,0.943500,0.943500,0.943500,0.9435
macro avg,0.942051,0.941310,0.941641,4000.0000
weighted avg,0.943479,0.943500,0.943452,4000.0000



Confusion Matrix


,Negatif,Netral,Positif
Negatif,1400,0,0
Netral,0,1077,123
Positif,1,102,1297



FP / FN


...


FP / FN


,Class,False Positive,False Negative
0,Negatif,0,0
1,Netral,0,205
2,Positif,205,0




RINGKASAN PERBANDINGAN


,Model,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1
0,Naive Bayes,0.94350,0.942051,0.941310,0.941641,0.943452
1,SVM,0.94950,0.949471,0.946627,0.947666,0.949372
2,LSTM,0.94875,0.957425,0.943056,0.946129,0.948105




FILE BERHASIL DISIMPAN
- classification_summary.csv
- *_classification_report.csv
- *_confusion_matrix.csv
- *_fp_fn.csv


**TAHAP 25 :** RESEARCH CONCLUSION

In [35]:
#Research Conclusion

print("=" * 70)
print("RESEARCH CONCLUSION")
print("=" * 70)

print("\n")
print("DATASET")
print("-" * 50)

print(f"Jumlah Dataset               : {len(df)}")
print(f"Jumlah Data Training         : {len(X_train)}")
print(f"Jumlah Data Testing          : {len(X_test)}")

print("\n")
print("BASELINE PERFORMANCE (SEED 42)")
print("-" * 50)

baseline_summary = pd.DataFrame({

    "Model": [

        "Naive Bayes",

        "SVM",

        "LSTM"

    ],

    "Accuracy": [

        accuracy_score(y_test, nb_pred),

        accuracy_score(y_test, svm_pred),

        accuracy_score(y_test_lstm, lstm_pred)

    ],

    "Precision": [

        precision_score(y_test, nb_pred, average="weighted"),

        precision_score(y_test, svm_pred, average="weighted"),

        precision_score(y_test_lstm, lstm_pred, average="weighted")

    ],

    "Recall": [

        recall_score(y_test, nb_pred, average="weighted"),

        recall_score(y_test, svm_pred, average="weighted"),

        recall_score(y_test_lstm, lstm_pred, average="weighted")

    ],

    "F1-score": [

        f1_score(y_test, nb_pred, average="weighted"),

        f1_score(y_test, svm_pred, average="weighted"),

        f1_score(y_test_lstm, lstm_pred, average="weighted")

    ]

})

display(baseline_summary)

print("\n")
print("AVERAGE PERFORMANCE (30 RANDOM STATE)")
print("-" * 50)

average_summary = pd.DataFrame({

    "Model": [

        "Naive Bayes",

        "SVM",

        "LSTM"

    ],

    "Average Accuracy": [

        np.mean(nb_accuracy),

        np.mean(svm_accuracy),

        np.mean(lstm_accuracy)

    ],

    "Std Accuracy": [

        np.std(nb_accuracy),

        np.std(svm_accuracy),

        np.std(lstm_accuracy)

    ]

})

average_summary = average_summary.sort_values(

    by="Average Accuracy",

    ascending=False

)

display(average_summary)

best_model = average_summary.iloc[0]

print("\n")

print(f"Model dengan rata-rata accuracy tertinggi : {best_model['Model']} ({best_model['Average Accuracy']:.4f})")

print("\n")
print("STATISTICAL EVALUATION")
print("-" * 50)

print("✓ Paired Student's T-Test")

print("✓ Wilcoxon Signed-Rank Test")

print("✓ McNemar Test")

print("\n")
print("ERROR ANALYSIS")
print("-" * 50)

print(f"Prediction Comparison     : {len(comparison_df)} data")

print(f"Prediction Difference     : {len(comparison_diff)} data")

print(f"Error Analysis            : {len(error_df)} data")

print(f"Top Error Cases           : {len(top_cases_df)} data")

print("\n")
print("IMPLEMENTATION DECISION")
print("-" * 50)

print("Model Implementasi Sistem : Support Vector Machine (SVM)")

print("\nAlasan")

print("- Memiliki performa baseline terbaik pada eksperimen utama.")

print("- Digunakan sebagai model implementasi pada Flask API.")

print("- Telah terintegrasi dengan sistem rekomendasi fashion.")

print("\n")
print("CATATAN EVALUASI")
print("-" * 50)

print("- Evaluasi statistik menunjukkan performa ketiga model sangat kompetitif.")

print("- LSTM memperoleh rata-rata akurasi tertinggi pada evaluasi berulang.")

print("- Hasil evaluasi statistik digunakan sebagai analisis tambahan.")

print("- Keputusan implementasi tetap mengacu pada eksperimen utama (baseline).")

print("\n")
print("=" * 70)
print("SEMUA EKSPERIMEN BERHASIL DISELESAIKAN")
print("=" * 70)

print("\nNotebook selesai.")

print("\nSeluruh file CSV berhasil dibuat.")

RESEARCH CONCLUSION


DATASET
--------------------------------------------------
Jumlah Dataset               : 20000
Jumlah Data Training         : 16000
Jumlah Data Testing          : 4000


BASELINE PERFORMANCE (SEED 42)
--------------------------------------------------


,Model,Accuracy,Precision,Recall,F1-score
0,Naive Bayes,0.94350,0.943479,0.94350,0.943452
1,SVM,0.94950,0.949982,0.94950,0.949372
2,LSTM,0.94875,0.955296,0.94875,0.948105




AVERAGE PERFORMANCE (30 RANDOM STATE)
--------------------------------------------------


,Model,Average Accuracy,Std Accuracy
2,LSTM,0.950200,0.004060
1,SVM,0.948558,0.003789
0,Naive Bayes,0.945458,0.003768




Model dengan rata-rata accuracy tertinggi : LSTM (0.9502)


STATISTICAL EVALUATION
--------------------------------------------------
✓ Paired Student's T-Test
✓ Wilcoxon Signed-Rank Test
✓ McNemar Test


ERROR ANALYSIS
--------------------------------------------------
Prediction Comparison     : 4000 data
Prediction Difference     : 246 data
Error Analysis            : 246 data
Top Error Cases           : 30 data


IMPLEMENTATION DECISION
--------------------------------------------------
Model Implementasi Sistem : Support Vector Machine (SVM)

Alasan
- Memiliki performa baseline terbaik pada eksperimen utama.
- Digunakan sebagai model implementasi pada Flask API.
- Telah terintegrasi dengan sistem rekomendasi fashion.


CATATAN EVALUASI
--------------------------------------------------
- Evaluasi statistik menunjukkan performa ketiga model sangat kompetitif.
- LSTM memperoleh rata-rata akurasi tertinggi pada evaluasi berulang.
- Hasil evaluasi statistik digunakan sebagai analisi